# **IMPORTANT QUESTION: What is State?**

- The **shared information** travelling throughout the graph.ie., from one node to the other node

# **Point 1 - What's the format to store Chat History?**

In [ ]:
!pip install -q -U langchain-openai

In [ ]:
import os
from getpass import getpass
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage,AIMessage

if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API Key: ")

messages = [HumanMessage(content=("My payment was deducted but my order is still pending.")),
            AIMessage(content=("I'll help you check your payment and order status.")),
            HumanMessage(content=("My order ID is ORD12345."))]

print("========== MESSAGE HISTORY ==========\n")
for message in messages:
    print("TYPE:",type(message).__name__)
    print("CONTENT:",message.content)
    print("-------------------------------------")

In [ ]:
# INITIALIZE CHAT MODEL
model = ChatOpenAI(model="gpt-4o-mini",temperature=0)
response = model.invoke(messages)

print("\n========== AI RESPONSE ==========\n")
print("TYPE:",type(response).__name__)
print("\nCONTENT:")
print(response.content)

# DISPLAY RESPONSE METADATA
print("\n========== RESPONSE METADATA ==========\n")
print(response.response_metadata)

# DISPLAY TOKEN USAGE
print("\n========== TOKEN USAGE ==========\n")
print(response.usage_metadata)

# **Point 2 - How to build a Tool Calling LLM using LangGraph?**

In [ ]:
!pip install -q -U langchain-openai langchain-core

In [ ]:
import os
from getpass import getpass
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage,AIMessage
from langchain_core.tools import tool

if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API Key: ")

@tool
def check_payment_status(order_id: str) -> str:
    """
    Check the payment status for a customer order.

    Args:
        order_id: Customer order ID.
    """
    if order_id == "ORD12345":
        return ("Payment successful.")

    return ("Payment information not found.")

@tool
def check_order_status(order_id: str) -> str:
    """
    Check the current order status.

    Args:
        order_id:Customer order ID.
    """
    if order_id == "ORD12345":
        return ("Order is currently pending.")

    return ("Order information not found.")


@tool
def create_support_ticket(issue: str) -> str:
    """
    Create a support ticket for a customer issue.

    Args:
       issue:Description of the issue.
    """
    return (f"Support ticket created for: {issue}")

tools = [check_payment_status,check_order_status,create_support_ticket]

In [ ]:
# INITIALIZE MODEL
model = ChatOpenAI(model="gpt-4o-mini",temperature=0)

# BIND TOOLS
model_with_tools = model.bind_tools(tools)

# USER QUESTION
messages = [HumanMessage(content=("My payment was deducted "
                                  "but my order is still pending. "
                                  "My order ID is ORD12345. "
                                  "Please check my payment status."))]

response = model_with_tools.invoke(messages)

print("\n========== MODEL RESPONSE ==========\n")
print(response)

# DISPLAY TOOL CALL
print("\n========== TOOL CALLS ==========\n")
print(response.tool_calls)

# **POINT 3 - Simple Router Agent using LangGraph**

In [ ]:
!pip install -q -U langchain-openai langgraph

import os
from getpass import getpass
from typing import TypedDict
from typing_extensions import Annotated
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain_core.messages import AnyMessage,HumanMessage
from langgraph.graph import StateGraph,START,END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode,tools_condition

if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API Key: ")

In [ ]:
class SupportState(TypedDict):
    messages: Annotated[list[AnyMessage],add_messages]

@tool
def check_payment_status(order_id: str) -> str:
    """
    Check the payment status for a customer order.
    Args:
        order_id: Customer order ID.
    """
    if order_id == "ORD12345":
        return ("Payment successful.")

    return ("Payment information not found.")

@tool
def check_order_status(order_id: str) -> str:
    """
    Check the current order status.
    Args:
        order_id:Customer order ID.
    """
    if order_id == "ORD12345":
        return ("Order is currently pending.")

    return ("Order information not found.")


@tool
def create_support_ticket(issue: str) -> str:
    """
    Create a support ticket for a customer issue.

    Args:
       issue:Description of the issue.
    """
    return (f"Support ticket created for: {issue}")

tools = [check_payment_status,check_order_status,create_support_ticket]

In [ ]:
# INITIALIZE CHAT MODEL
model = ChatOpenAI(model="gpt-4o-mini",temperature=0)

# 9. BIND TOOLS TO MODEL
model_with_tools = model.bind_tools(tools)

In [ ]:
# CREATE AGENT NODE
def agent_node(state: SupportState):
    print("\n🤖 AGENT NODE")
    response = model_with_tools.invoke(state["messages"])
    return {"messages": [response]}

# CREATE ToolNode
tool_node = ToolNode(tools)

In [ ]:
# CREATE STATE GRAPH
graph = StateGraph(SupportState)

graph.add_node("agent",agent_node)
graph.add_node("tools",tool_node)

graph.add_edge(START,"agent")

#no need to create contional router to give access to tools
graph.add_conditional_edges("agent",tools_condition,{"tools": "tools",
                                                      END: END})

graph.add_edge("tools","agent")

In [ ]:
app = graph.compile()

from IPython.display import Image, display
display(Image(app.get_graph().draw_mermaid_png()))

In [ ]:
# RUN AGENT
result = app.invoke({"messages": [HumanMessage(content=("My payment was deducted "
                                                        "but my order is still pending. "
                                                        "My order ID is ORD12345."))]})
result

In [ ]:
# DISPLAY CONVERSATION
for message in result["messages"]:
    message.pretty_print()

# **THE END.**